In [1]:
%pip install boto3
%pip install scikit-learn

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import boto3

# Initialize the S3 client 
s3 = boto3.client('s3')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Preresequite: Create `.env` file in this dir, then add:
- `AWS_ACCESS_KEY_ID="Your Key"`, 
- `AWS_SECRET_ACCESS_KEY="Your Secrete Key"`, 
- `AWS_DEFAULT_REGION="us-east-2"`

Note: This file is in the `.gitignore` so will need to look at AWS IAM for keys or contact maintainers of repo for access.

In [ ]:
import boto3
import pandas as pd
import io
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

# 1. Initialize the S3 client
s3 = boto3.client('s3')
bucket_name = 'cleaned-filtered-data'
file_key = 'V-Dem/V_Dem_Filtered.csv' 

# 2. Fetch the object
obj = s3.get_object(Bucket=bucket_name, Key=file_key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Loaded: {file_key}")
print(f"Columns available: {df.columns.tolist()}")

Loaded: V-Dem/V_Dem_Filtered.csv
Columns available: ['year', 'country_name', 'country_id', 'country_text_id', 'v2x_accountability', 'v2x_civlib', 'v2x_corr', 'v2x_electoral_integrity', 'v2x_polyarchy', 'v2x_regime', 'e_v2xel_locelec_5C', 'e_wbgi_gee', 'e_uds', 'e_gdp', 'e_gdppc', 'e_miinflat']


In [ ]:
 
# 1. Filter for Modern Era (Post-WWII)
df_modern = df[df['year'] >= 1945].copy()
df_modern = df_modern.sort_values(['country_id', 'year'])

# 2. Calculate 3-year change in Polyarchy (the more reliable score)
main_score = 'v2x_polyarchy'
df_modern['dem_change_3yr'] = df_modern.groupby('country_id')[main_score].shift(-3) - df_modern[main_score]

# 3. Create Target 1: Future GDP
df_modern['target_gdp_3yr'] = df_modern.groupby('country_id')['e_gdppc'].shift(-3)

# 4. Create Target 2: Backsliding (Bottom 5th Percentile of crashes)
df_clean = df_modern.dropna(subset=[main_score, 'dem_change_3yr', 'e_gdppc', 'target_gdp_3yr'])
threshold = df_clean['dem_change_3yr'].quantile(0.05)
df_clean['target_backslide_3yr'] = (df_clean['dem_change_3yr'] <= threshold).astype(int)

print(f"Dataset ready. Total records: {len(df_clean)}")
print(f"Backsliding cases identified: {df_clean['target_backslide_3yr'].sum()}")

Dataset ready for training. Total records: 21910


In [ ]:
from sklearn.model_selection import cross_val_score

# 1. Define Features (X) and Target (y)
X_gdp = df_clean.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 'dem_change_3yr',
    'e_gdppc', 'e_gdp', 'e_miinflat', 'e_uds'
])
y_gdp = df_clean['target_gdp_3yr'] # <--- THIS DEFINES 'Y'

# 2. Train the Robust Model
gdp_model = RandomForestRegressor(
    n_estimators=500, 
    max_depth=10, 
    min_samples_leaf=10, 
    random_state=42
)

# 3. Check for the "Honest" Score
cv_scores = cross_val_score(gdp_model, X_gdp, y_gdp, cv=5)
gdp_model.fit(X_gdp, y_gdp)

print(f"Honest Average R² Score: {cv_scores.mean():.2f}")
print("\nVerified GDP Red Flags:")
print(pd.Series(gdp_model.feature_importances_, index=X_gdp.columns).sort_values(ascending=False).head(5))

NameError: name 'y' is not defined

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Features and Target
X_back = df_clean.drop(columns=[
    'year', 'country_name', 'country_id', 'country_text_id', 
    'target_gdp_3yr', 'target_backslide_3yr', 'dem_change_3yr',
    'v2x_polyarchy', 'e_uds', 'v2x_regime'
])
y_back = df_clean['target_backslide_3yr']

# 2. Train with Balanced Weights
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_back, y_back, test_size=0.2, random_state=42, stratify=y_back
)
back_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
back_model.fit(X_train_b, y_train_b)

# 3. Results
print("Backsliding Prediction Report:")
print(classification_report(y_test_b, back_model.predict(X_test_b)))

Backsliding Prediction Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4382

    accuracy                           1.00      4382
   macro avg       1.00      1.00      1.00      4382
weighted avg       1.00      1.00      1.00      4382

